ModuleNotFoundError: No module named 'kagglehub'

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

Using device: cuda


In [22]:
import os
DATA_DIR = "..\\real_vs_fake_face_dataset\\real_vs_fake\\real-vs-fake"

for dirs in ["train", "valid", "test"]:
    path = f"{DATA_DIR}\\{dirs}"
    real_count = len(os.listdir(f"{path}\\real"))
    fake_count = len(os.listdir(f"{path}\\fake"))
    print(f"{dirs}: {real_count} real | {fake_count} fake")

train: 50000 real | 50000 fake
valid: 10000 real | 10000 fake
test: 10000 real | 10000 fake


In [23]:
import os
DATA_DIR = "../real_vs_fake_face_dataset/real_vs_fake/real-vs-fake"

for dirs in ["train", "valid", "test"]:
    path = f"{DATA_DIR}/{dirs}"
    real_count = len(os.listdir(f"{path}/real"))
    fake_count = len(os.listdir(f"{path}/fake"))
    print(f"{dirs}: {real_count} real | {fake_count} fake")

train: 50000 real | 50000 fake
valid: 10000 real | 10000 fake
test: 10000 real | 10000 fake


In [24]:
# import subprocess, sys
# subprocess.check_call([
#     sys.executable, "-m", "pip", "install",
#     "torch>=2.4", "torchvision>=0.19", "--upgrade", "-q"
# ])

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

# ── Config ──────────────────────────────────────────────
DATA_DIR = "../real_vs_fake_face_dataset/real_vs_fake/real-vs-fake"
BATCH_SIZE = 32
EPOCHS     = 10
LR         = 3e-4
IMG_SIZE   = 224
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

# ── Transforms ──────────────────────────────────────────
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),                      # reduces overfitting
    transforms.ColorJitter(brightness=0.2, contrast=0.2),   # accounts for lighting changes
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],   # ImageNet mean
                         [0.229, 0.224, 0.225]),   # ImageNet std
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# ── Datasets & Loaders ──────────────────────────────────
train_ds = datasets.ImageFolder(f"{DATA_DIR}/train", transform=train_transforms)
val_ds   = datasets.ImageFolder(f"{DATA_DIR}/valid", transform=val_transforms)
test_ds  = datasets.ImageFolder(f"{DATA_DIR}/test",  transform=val_transforms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Classes: {train_ds.classes}")   # ['fake', 'real']

# ── Model ───────────────────────────────────────────────
model = models.efficientnet_b0(weights="IMAGENET1K_V1")
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
model = model.to(DEVICE)

# ── Training Setup ──────────────────────────────────────
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

# ── Train & Eval Functions ──────────────────────────────
def train_epoch(model, loader):
    model.train()
    total_loss, correct = 0, 0
    for imgs, labels in tqdm(loader, desc="Training"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def eval_epoch(model, loader):
    model.eval()
    total_loss, correct = 0, 0
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Evaluating"):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    auc = roc_auc_score(all_labels, all_probs)
    return total_loss / len(loader), correct / len(loader.dataset), auc

# ── Training Loop ───────────────────────────────────────
best_auc = 0
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    train_loss, train_acc = train_epoch(model, train_loader)
    val_loss, val_acc, val_auc = eval_epoch(model, val_loader)
    scheduler.step()

    print(f"Train — Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
    print(f"Val   — Loss: {val_loss:.4f}  | Acc: {val_acc:.4f} | AUC: {val_auc:.4f}")

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  ✅ Saved best model (AUC: {best_auc:.4f})")

# ── Final Test Evaluation ────────────────────────────────
model.load_state_dict(torch.load("best_model.pth"))
test_loss, test_acc, test_auc = eval_epoch(model, test_loader)
print(f"\nTest Results — Acc: {test_acc:.4f} | AUC: {test_auc:.4f}")

Using device: cuda
Classes: ['fake', 'real']


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\TUF GAMING F15/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:00<00:00, 25.1MB/s]



Epoch 1/10


Evaluating: 100%|██████████| 625/625 [01:32<00:00,  6.78it/s]


Train — Loss: 0.0684 | Acc: 0.9733
Val   — Loss: 0.0199  | Acc: 0.9920 | AUC: 0.9998
  ✅ Saved best model (AUC: 0.9998)

Epoch 2/10


...

Train — Loss: 0.0133 | Acc: 0.9954
Val   — Loss: 0.0034  | Acc: 0.9989 | AUC: 1.0000
  ✅ Saved best model (AUC: 1.0000)

Epoch 4/10


Evaluating: 100%|██████████| 625/625 [01:19<00:00,  7.82it/s]


Train — Loss: 0.0096 | Acc: 0.9969
Val   — Loss: 0.0065  | Acc: 0.9976 | AUC: 1.0000

Epoch 5/10


Training:   0%|          | 0/3125 [00:06<?, ?it/s]


KeyboardInterrupt: 